# Part 5: Connect KB to a Foundry agent

In the previous parts, you built a knowledge base with indexed and MCP sources, and consumed it as an MCP endpoint. In this final part, you'll connect the knowledge base to a **Microsoft Foundry Agent** — enabling conversational, multi-turn interactions grounded in your enterprise data.

The Foundry Agent Service uses the knowledge base in `EXTRACTIVE_DATA` mode (raw content chunks instead of synthesized answers) and wraps it as an MCP tool that the agent can call during conversation.

> **📋 Prerequisites**: This notebook requires a Microsoft Foundry project. The environment variables `PROJECT_ENDPOINT`, `PROJECT_RESOURCE_ID`, and `PROJECT_CONNECTION_NAME` must be set in your `.env` file.

## Step 0: Sign in to Azure CLI

The Foundry Agent SDK uses your Azure CLI identity. Run the cell below and follow the browser prompt to sign in as your lab user.

In [ ]:
!az login

## Step 1: Load environment variables

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

# Azure AI Search configuration
search_endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
search_api_key = os.environ["AZURE_SEARCH_ADMIN_KEY"]
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_openai_key = os.environ["AZURE_OPENAI_KEY"]
azure_openai_chatgpt_deployment = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
azure_openai_chatgpt_model_name = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]

# Foundry project configuration
project_endpoint = os.environ["PROJECT_ENDPOINT"]
project_resource_id = os.environ["PROJECT_RESOURCE_ID"]
project_connection_name = os.environ["PROJECT_CONNECTION_NAME"]

agent_name = "kb-grounded-agent"
knowledge_base_name = "foundry-agent-knowledge-base"
api_version = "2025-11-01-preview"

print("Environment variables loaded")

## Step 2: Re-create the knowledge base with extractive data mode

Foundry Agents require the knowledge base to use `extractiveData` output mode (not `answerSynthesis`). In this mode, the KB returns raw content chunks that the agent can reason over, rather than a pre-synthesized answer.

We also use `KnowledgeRetrievalMinimalReasoningEffort` for efficient extraction — Foundry Agent Service handles the final reasoning.

In [ ]:
import requests

rest_headers = {"Content-Type": "application/json", "api-key": search_api_key}

# Ensure indexed knowledge sources exist
for ks_name, index_name, description in [
    ("healthdocs-knowledge-source", "healthdocs", "Zava health documents"),
    ("hrdocs-knowledge-source", "hrdocs", "Zava HR documents"),
]:
    ks_body = {
        "name": ks_name,
        "kind": "searchIndex",
        "description": description,
        "searchIndexParameters": {
            "searchIndexName": index_name,
            "sourceDataFields": [{"name": "blob_path"}, {"name": "snippet"}],
            "searchFields": [{"name": "snippet"}]
        }
    }
    url = f"{search_endpoint}/knowledgesources/{ks_name}?api-version={api_version}"
    response = requests.put(url, json=ks_body, headers=rest_headers)
    response.raise_for_status()
    print(f"Knowledge source '{ks_name}' ready: {response.status_code}")

# Create KB with extractive data mode
kb_body = {
    "name": knowledge_base_name,
    "description": "Knowledge base for Foundry Agent grounding with extractive data",
    "outputMode": "extractiveData",
    "knowledgeSources": [
        {"name": "healthdocs-knowledge-source"},
        {"name": "hrdocs-knowledge-source"}
    ],
    "models": [
        {
            "kind": "azureOpenAI",
            "azureOpenAIParameters": {
                "resourceUri": azure_openai_endpoint,
                "deploymentId": azure_openai_chatgpt_deployment,
                "modelName": azure_openai_chatgpt_model_name,
                "apiKey": azure_openai_key
            }
        }
    ],
    "retrievalReasoningEffort": {"kind": "minimal"}
}

url = f"{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}"
response = requests.put(url, json=kb_body, headers=rest_headers)
response.raise_for_status()
print(f"\nKnowledge base '{knowledge_base_name}' created with extractiveData mode: {response.status_code}")

## Step 3: Build the MCP endpoint URL

The Foundry Agent connects to the knowledge base via its MCP endpoint. Build the URL:

In [ ]:
mcp_endpoint = f"{search_endpoint}/knowledgebases/{knowledge_base_name}/mcp?api-version=2025-11-01-Preview"

print("MCP endpoint for Foundry Agent:")
print(mcp_endpoint)

## Step 4: Create the Foundry project connection

The Foundry Agent needs a **project connection** to reach the KB's MCP endpoint. This is created via the Azure Resource Manager REST API.

The connection uses `ProjectManagedIdentity` for auth and `RemoteTool` as the category, which tells Foundry this is an external tool endpoint.

In [ ]:
from azure.identity import AzureCliCredential, get_bearer_token_provider

arm_credential = AzureCliCredential(process_timeout=60)
bearer_token_provider = get_bearer_token_provider(
    arm_credential,
    "https://management.azure.com/.default",
)

connection_url = (
    f"https://management.azure.com{project_resource_id}"
    f"/connections/{project_connection_name}?api-version=2025-10-01-preview"
)

connection_body = {
    "name": project_connection_name,
    "type": "Microsoft.MachineLearningServices/workspaces/connections",
    "properties": {
        "authType": "ProjectManagedIdentity",
        "category": "RemoteTool",
        "target": mcp_endpoint,
        "isSharedToAll": True,
        "audience": "https://search.azure.com/",
        "metadata": {"ApiType": "Azure"},
    },
}

arm_headers = {
    "Authorization": f"Bearer {bearer_token_provider()}",
    "Content-Type": "application/json",
}

response = requests.put(connection_url, json=connection_body, headers=arm_headers)
response.raise_for_status()

print(f"Project connection '{project_connection_name}' created: {response.status_code}")

## Step 5: Create a Foundry agent with MCPTool

Create an agent using the `azure-ai-projects` SDK. The agent uses an `MCPTool` that points to the knowledge base connection. Set `require_approval="never"` so the agent can call the Azure AI Search KB endpoint through the project connection, and restrict it to `knowledge_base_retrieve`.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.identity import AzureCliCredential

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=AzureCliCredential(process_timeout=60)
)

agent_instructions = """
You are a helpful assistant that must use the knowledge base to answer all questions.
You must never answer from your own knowledge under any circumstances.
Every answer must include citations in the format: 【message_idx:search_idx†source_name】
If the knowledge base does not contain the answer, respond with "I don't know".
""".strip()

mcp_tool = MCPTool(
    server_label="knowledge-base",
    server_url=mcp_endpoint,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=project_connection_name,
)

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=azure_openai_chatgpt_deployment,
        instructions=agent_instructions,
        tools=[mcp_tool],
    ),
)

agent_reference = {
    "name": agent.name,
    "version": agent.version,
    "type": "agent_reference",
}

print(f"Agent '{agent.name}' created with ID: {agent.id} and version: {agent.version}")

## Step 6: Chat with the agent

Send a question to the agent. The agent uses `tool_choice="required"` to always consult the knowledge base before answering.

In [ ]:
from IPython.display import Markdown, display

openai_client = project_client.get_openai_client()
conversation = openai_client.conversations.create()

response = openai_client.responses.create(
    conversation=conversation.id,
    tool_choice="required",
    input=[
        {
            "role": "user",
            "content": "What is the responsibility of the Zava CEO and what health plans are available?",
        }
    ],
    extra_body={"agent_reference": agent_reference},
)
display(Markdown(response.output_text))

## Step 7: Multi-turn conversation

The agent supports multi-turn conversations. Ask a follow-up question that builds on the previous answer:

In [ ]:
response = openai_client.responses.create(
    conversation=conversation.id,
    tool_choice="required",
    input=[
        {
            "role": "user",
            "content": "Which of those health plans has the best coverage for mental health services?",
        }
    ],
    extra_body={"agent_reference": agent_reference},
)

display(Markdown(response.output_text))

## Summary

You've connected your knowledge base to a Foundry agent. The agent uses the KB's MCP endpoint as a grounding tool, enabling conversational, multi-turn interactions over your enterprise data.

**Key concepts:**
- Foundry Agents require `extractiveData` output mode, not `answerSynthesis`
- Use `KnowledgeRetrievalMinimalReasoningEffort` so the agent handles the final reasoning
- A project connection links the Foundry project to the KB's MCP endpoint and carries the Search audience for managed-identity auth
- `MCPTool` with `require_approval="never"` and `allowed_tools=["knowledge_base_retrieve"]` keeps the tool path focused on KB retrieval
- `tool_choice="required"` ensures the agent always consults the KB before answering
- The Responses API supports multi-turn chat by reusing the same conversation with new `input` turns

> **Note**: Foundry Agent Service can't send per-request custom headers to MCP tools. This notebook works because Azure AI Search KB MCP uses a project connection with project managed identity instead of user-specific runtime headers.

## 🎉 Lab complete!

You've completed all 5 parts of the lab. Here's what you built:

1. **Multi-source knowledge base** with indexed HR and health documents
2. **KB as MCP endpoint** for developer tools like GitHub Copilot CLI
3. **Live MCP source** (Microsoft Learn) for real-time documentation access
4. **Authenticated MCP source** (GitHub) with Bearer token at query time
5. **Foundry agent** grounded on your knowledge base for conversational AI